# 分类回归实战_预处理标准化

实战步骤：

1、加载数据集，理解数据集 fashion_minst

2、搭建 class NeuralNetwork 模型

3、设置交叉熵损失函数，SGD 优化器

4、编写评估函数 evaluating

5、编写训练函数 train

6、开始训练 训练多少次

7、绘制损失、准确率曲线

## 1. 数据集预处理

In [1]:
# 计算数据集的均值和标准差
import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

# 首先加载数据（仅转换为Tensor，不进行其他预处理）
temp_transform = transforms.ToTensor()
temp_train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=temp_transform)
temp_train_loader = DataLoader(temp_train_dataset, batch_size=64, shuffle=False)

# 初始化累加器
sum_pixels = 0.0
sum_squared_pixels = 0.0
total_pixels = 0

# 遍历所有数据计算均值和平方均值
print("正在计算训练集的均值和标准差...")
for images, _ in temp_train_loader:
    # images形状: [batch_size, 1, 28, 28]
    # 将所有像素值展平为一维
    pixels = images.view(-1)  # [batch_size * 1 * 28 * 28]
    
    # 累加所有像素值和像素值的平方
    sum_pixels += pixels.sum().item()
    sum_squared_pixels += (pixels ** 2).sum().item()
    total_pixels += pixels.numel()

# 计算总体均值 μ = (1/n) * Σ(x_i)
mean = sum_pixels / total_pixels

# 根据公式计算方差：σ² = (1/n) * Σ(x_i²) - μ²
mean_squared = sum_squared_pixels / total_pixels
variance = mean_squared - mean ** 2

# 计算标准差 σ = √σ²
std = variance ** 0.5

print(f"总像素数: {total_pixels}")
print(f"训练集均值 (μ): {mean:.6f}")
print(f"训练集标准差 (σ): {std:.6f}")
print(f"训练集方差 (σ²): {variance:.6f}")
print(f"\n验证计算:")
print(f"  (1/n) * Σ(x_i²) = {mean_squared:.6f}")
print(f"  μ² = {mean**2:.6f}")
print(f"  σ² = (1/n) * Σ(x_i²) - μ² = {mean_squared:.6f} - {mean**2:.6f} = {variance:.6f}")

# 保存均值和标准差供后续使用
dataset_mean = mean
dataset_std = std


正在计算训练集的均值和标准差...
总像素数: 47040000
训练集均值 (μ): 0.286041
训练集标准差 (σ): 0.353024
训练集方差 (σ²): 0.124626

验证计算:
  (1/n) * Σ(x_i²) = 0.206445
  μ² = 0.081819
  σ² = (1/n) * Σ(x_i²) - μ² = 0.206445 - 0.081819 = 0.124626


In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

# 定义数据预处理：将图像转换为Tensor，并进行标准化
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为Tensor并归一化到[0, 1]
    transforms.Normalize(mean=[dataset_mean], std=[dataset_std])  # 标准化
])

# 加载数据集
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

# 定义DataLoader参数
batch_size = 64

# 创建DataLoader对数据进行分批
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 示例：打印一个batch的形状
images, labels = next(iter(train_loader))
print("训练集一个batch的图像形状:", images.shape)
print("训练集一个batch的标签形状:", labels.shape)
print(f"标准化后的数据统计:")
print(f"  均值: {images.mean().item():.6f}")
print(f"  标准差: {images.std().item():.6f}")
print(f"  最小值: {images.min().item():.6f}")
print(f"  最大值: {images.max().item():.6f}")

print("\n第一个batch中的第一个tensor（标准化后）：")
print(images[0])



训练集一个batch的图像形状: torch.Size([64, 1, 28, 28])
训练集一个batch的标签形状: torch.Size([64])
标准化后的数据统计:
  均值: -0.005083
  标准差: 0.991044
  最小值: -0.810258
  最大值: 2.022409

第一个batch中的第一个tensor（标准化后）：
tensor([[[-0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103],
         [-0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103],
         [-0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103, -0.8103,
          -0

## 2. 搭建模型

In [3]:
from torch import nn

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()  # 展平输入，[batch, 1, 28, 28] -> [batch, 28*28]
        self.fc1 = nn.Linear(28*28, 300)  # 第一全连接层，将784维输入映射到300维
        self.relu1 = nn.ReLU()  # 第一ReLU激活函数
        self.fc2 = nn.Linear(300, 100)  # 第二全连接层，将300维输入映射到100维
        self.relu2 = nn.ReLU()  # 第二ReLU激活函数
        self.fc3 = nn.Linear(100, 10)  # 输出层，将100维输入映射到10个类别

    # 前向传播方法，调用 model(x) 会自动调用 forward(x)
    def forward(self, x):
        x = self.flatten(x)
        # print("经过flatten后形状:", x.shape)
        x = self.fc1(x)
        # print("经过fc1后形状:", x.shape)
        x = self.relu1(x)
        # print("经过relu1后形状:", x.shape)
        x = self.fc2(x)
        # print("经过fc2后形状:", x.shape)
        x = self.relu2(x)
        # print("经过relu2后形状:", x.shape)
        x = self.fc3(x)
        # print("经过fc3后形状:", x.shape)
        return x


In [4]:
# 模型测试

# 随机生成一个数据，形状为 (1, 1, 28, 28)
random_input = torch.randn(1, 1, 28, 28)

# 实例化模型
model = NeuralNetwork()

# 调用模型进行前向传播
output = model(random_input)

print("模型输出：")
print(output)


模型输出：
tensor([[ 0.0715,  0.1801, -0.0169, -0.0684, -0.1166, -0.0326,  0.0033,  0.0225,
          0.2297, -0.1114]], grad_fn=<AddmmBackward0>)


## 3. 封装Trainer类进行训练与评估

In [ ]:
# 导入 Trainer 模块
from Train import Trainer
import torch.optim as optim

# 创建新的模型实例（如果之前已经训练过，可以创建新模型重新训练）
model_with_trainer = NeuralNetwork()

# 创建 Trainer 实例，传入模型和数据加载器
trainer = Trainer(
    model=model_with_trainer,
    train_loader=train_loader,
    val_loader=test_loader
)

# 开始训练（使用 Trainer 类）
num_epochs = 40
trainer.train(num_epochs=num_epochs)

# 使用 Trainer 进行最终评估
print("\n" + "=" * 60)
print("最终模型评估")
print("=" * 60)
final_loss, final_acc = trainer.evaluating()
print(f"最终测试损失: {final_loss:.4f}")
print(f"最终测试准确率: {final_acc:.4f}")

# 使用 Trainer 绘制训练历史曲线
trainer.plot_history()



Trainer 初始化完成
设备: cuda
损失函数: CrossEntropyLoss()
优化器: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)
训练集: 60000 个样本, 938 个批次
验证集: 10000 个样本, 157 个批次

开始训练模型
训练轮数: 40
训练集: 60000 个样本, 938 个批次
验证集: 10000 个样本, 157 个批次
设备: cuda
优化器: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    momentum: 0.9
    nesterov: False
    weight_decay: 0
)

Epoch [1/40], Train Loss: 0.5314, Train Acc: 0.8087, Val Loss: 0.4293, Val Acc: 0.8432
Epoch [2/40], Train Loss: 0.3605, Train Acc: 0.8687, Val Loss: 0.3741, Val Acc: 0.8649
Epoch [3/40], Train Loss: 0.3217, Train Acc: 0.8809, Val Loss: 0.3646, Val Acc: 0.8684
Epoch [4/40], Train Loss: 0.2931, Train Acc: 0.8916, Val Loss: 0.3606, Val Acc: 0.8716
Epoch [5/40], Train Loss: 0.2744, Train Acc: 0.8973, Val Loss: 0.3342, Val Acc: 0.8

## 4. 训练结果对比分析：标准化 vs 非标准化

### 实验设置对比

| 项目 | 无标准化 | 有标准化 |
|------|---------|---------|
| 数据预处理 | `transforms.ToTensor()` | `transforms.ToTensor()` + `transforms.Normalize()` |
| 数据集均值 (μ) | - | 0.286041 |
| 数据集标准差 (σ) | - | 0.353024 |
| 数据范围 | [0, 1] | 标准化后均值≈0，标准差≈1 |
| 模型结构 | 相同 | 相同 |
| 优化器 | SGD (lr=0.01, momentum=0.9) | SGD (lr=0.01, momentum=0.9) |
| 训练轮数 | 40 | 40 |
| Batch Size | 64 | 64 |


### 详细分析

#### 1. **收敛速度分析**

**第一轮训练表现：**
- **无标准化**: 训练损失 0.6884，训练准确率 76.02%，验证准确率 81.32%
- **有标准化**: 训练损失 0.5257，训练准确率 81.14%，验证准确率 84.43%

**结论**: 标准化方法在训练初期就表现出明显的优势：
- 训练损失降低了 **23.6%** (0.1627)
- 训练准确率提升了 **5.12%** (从 76.02% 到 81.14%)
- 验证准确率提升了 **3.11%** (从 81.32% 到 84.43%)

这说明标准化能够：
- 加速模型收敛
- 提高训练初期的学习效率
- 使梯度更新更加稳定

#### 2. **最终性能对比**

**训练集表现：**
- **无标准化**: 训练损失 0.1225，训练准确率 95.38%
- **有标准化**: 训练损失 0.0601，训练准确率 97.79%

**验证集/测试集表现：**
- **无标准化**: 验证损失 0.3633，验证准确率 88.74%，测试准确率 88.74%
- **有标准化**: 验证损失 0.5445，验证准确率 89.22%，测试准确率 89.22%

**关键发现：**
1. ✅ **训练准确率显著提升**: 从 95.38% 提升到 97.79% (+2.41%)
2. ✅ **测试准确率略有提升**: 从 88.74% 提升到 89.22% (+0.48%)
3. ⚠️ **验证损失增加**: 从 0.3633 增加到 0.5445 (+49.8%)

#### 3. **过拟合现象分析**

**训练损失 vs 验证损失：**
- **无标准化**: 训练损失 0.1225，验证损失 0.3633，差距 0.2408
- **有标准化**: 训练损失 0.0601，验证损失 0.5445，差距 0.4844

**分析：**
- 标准化方法在训练集上表现更好（损失更低，准确率更高）
- 但训练损失和验证损失之间的差距更大，表明存在**过拟合**现象
- 这可能是因为标准化使模型能够更快速地学习训练数据的特征，但泛化能力需要进一步优化

#### 4. **标准化带来的优势**

1. **加速收敛**: 
   - 数据标准化后，所有特征都在相似的尺度上，使得梯度下降算法能够更快地找到最优解
   - 第一轮训练就达到了更好的性能

2. **提高训练稳定性**:
   - 标准化后的数据均值接近0，标准差接近1，使得梯度更新更加稳定
   - 避免了不同特征尺度差异导致的训练不稳定问题

3. **更好的特征学习**:
   - 标准化使模型能够更有效地学习数据的特征表示
   - 训练准确率从 95.38% 提升到 97.79%

#### 5. **需要注意的问题**

1. **过拟合风险**:
   - 标准化方法虽然提高了训练准确率，但验证损失增加，表明可能存在过拟合
   - 建议考虑使用正则化技术（如 Dropout、权重衰减等）来改善泛化能力

2. **训练时间**:
   - 虽然需要额外计算数据集的均值和标准差，但这是一次性计算，对整体训练时间影响很小

#### 6. **总结与建议**

**标准化方法的优势：**
- ✅ 显著提升训练准确率 (+2.41%)
- ✅ 提升测试准确率 (+0.48%)
- ✅ 加速模型收敛
- ✅ 提高训练稳定性

**改进建议：**
- 考虑添加 Dropout 层来减少过拟合
- 可以尝试早停（Early Stopping）策略
- 可以调整学习率或使用学习率调度器
- 考虑使用数据增强技术进一步提升泛化能力

**结论：**
标准化是深度学习中非常重要的数据预处理步骤，在本实验中显著提升了模型的训练效果和收敛速度。虽然存在一定的过拟合现象，但通过适当的正则化技术可以进一步优化模型的泛化性能。
